# spd1305x.py bench test

Manual test of the SPD1305X driver against real hardware -- confirms `turn_on()`, `set_voltage()`, `set_current_limit()`, `output_on()`/`output_off()` actually work before trusting them inside `cw_odmr_lock_in.py`'s automated flow.

**Before running**: connect the SPD1305X's output to a dummy load or the actual amplifier, not left open/shorted carelessly -- use low voltage/current for the very first test if you're not sure the SCPI syntax below matches this unit yet (see `spd1305x.py`'s module docstring -- NOT YET VERIFIED against real hardware).

## Find the resource address

In [1]:
import pyvisa
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('USB0::0x0957::0x5707::MY53800810::INSTR', 'USB0::0xF4EC::0x1410::SPD13DCD7R1877::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL7::INSTR', 'ASRL8::INSTR', 'GPIB0::18::INSTR', 'GPIB1::19::INSTR', 'GPIB2::2::INSTR')


Copy the SPD1305X's resource string from the list above and paste it below (replacing `PSU_RESOURCE`) if it doesn't match `cw_odmr_lock_in.py`'s current `PSU_RESOURCE` placeholder.

In [2]:
import sys
sys.path.insert(0, r"C:\Users\codin\Documents\projects\diamonds\experiment")

from spd1305x import SPD1305X

PSU_RESOURCE = "USB0::0xF4EC::0x1410::SPD13DCD7R1877::INSTR"  # update if needed

psu = SPD1305X(PSU_RESOURCE, debug=True)
print(psu.idn())

SPD1305X: connected
Siglent Technologies,SPD1305X,SPD13DCD7R1877,2.1.2.11,V2.0


## Test set_voltage() / set_current_limit() individually

Output stays OFF for these -- just confirms the setpoint commands are accepted (debug=True prints each SCPI command; `write()` raises if the instrument reports an error).

In [3]:
psu.set_voltage(5.0)
print("voltage setpoint readback:", psu.get_voltage_setpoint())

psu.set_current_limit(0.5)
print("current limit setpoint readback:", psu.get_current_limit_setpoint())

VOLT 5.0 => +0, No error
voltage setpoint readback: 5.0
CURR 0.5 => +0, No error
current limit setpoint readback: 0.5


## Test output_on() / output_off()

Confirm the measured output actually follows -- checks `read_voltage()`/`read_current()` before and after toggling. With nothing/a high-impedance load connected, expect ~0 measured current.

In [4]:
print("before output_on: V =", psu.read_voltage(), "I =", psu.read_current())

psu.output_on()
import time
time.sleep(0.5)
print("after output_on:  V =", psu.read_voltage(), "I =", psu.read_current())

before output_on: V = 0.0 I = 0.0
OUTP CH1,ON => +0, No error
after output_on:  V = 4.999 I = 0.0


In [5]:
psu.output_off()
time.sleep(0.5)
print("after output_off: V =", psu.read_voltage(), "I =", psu.read_current())

OUTP CH1,OFF => +0, No error
after output_off: V = 0.0 I = 0.0


## Test turn_on() convenience method

Sets voltage + current limit BEFORE enabling output, then enables it -- this is what `cw_odmr_lock_in.py` actually calls. Using the real 12 V / 1.9 A amplifier-supply values here -- make sure the amplifier (or an appropriate load) is actually connected before running this cell.

In [6]:
psu.turn_on(12.0, 1.9)
time.sleep(0.5)
print("voltage setpoint:", psu.get_voltage_setpoint())
print("current limit setpoint:", psu.get_current_limit_setpoint())
print("measured: V =", psu.read_voltage(), "I =", psu.read_current())

VOLT 12.0 => +0, No error
CURR 1.9 => +0, No error
OUTP CH1,ON => +0, No error
SPD1305X: output ON at 12.0 V, 1.9 A limit
voltage setpoint: 12.0
current limit setpoint: 1.9
measured: V = 11.998 I = 0.0


## Shut down

In [7]:
psu.turn_off()
time.sleep(0.2)
print("after turn_off: V =", psu.read_voltage(), "I =", psu.read_current())

psu.go_to_local()
psu.close()
print("closed")

OUTP CH1,OFF => +0, No error
SPD1305X: output OFF
after turn_off: V = 0.0 I = 0.0
SYST:LOCAL => -113,Undefined header,SYST:LOCAL


RuntimeError: SPD1305X error after 'SYST:LOCAL': -113,Undefined header,SYST:LOCAL